# 3.1 資料與時點處理

## 資料來源

| 資料類別 | 來源 | 範圍 |
| :--- | :--- | :--- |
| **價格** | Tiingo 日頻調整後收盤價 | S&P 500 成分股，2000-01 – 2025-12 |
| **基本面** | SEC EDGAR XBRL `companyfacts` | 公司財報概念，逐季 |
| **產業分類** | GICS | 十一大類（Sector 層級） |

**樣本規模**　資料庫共 **843 檔**曾任成分股者；每一形成期約 **600 檔**在冊；
交易日合計 **6,287 日**（≈ 24.9 年）。


## 三處時點（point-in-time）控制

回測的每一個決策，只能使用**該決策時點之前已公開**的資訊。
本研究於三處施加此限制：

| 環節 | 控制方式 | 所防範的偏誤 |
| :--- | :--- | :--- |
| **成分股認定** | 依指數成分股的**歷史異動紀錄**（納入日與剔除日），每期僅取當時確實在指數內者；樣本含 **170 檔**其後已下市者 | **存活者偏誤**（survivorship bias） |
| **基本面對齊** | 一律取**申報日不晚於形成期結束日**的最新一筆，而非以財報**所屬期末日**對齊 | **前視偏誤**（look-ahead bias）：財報期末日與實際公開日之間有數週至數月落差 |
| **特徵計算窗** | 僅使用形成期窗內資料；交易期價格不參與任何估計或配適 | **前視偏誤** |

::: {.callout-note}

### 為何「申報日」而非「期末日」

一家公司 2020 年第一季（期末 3/31）的財報，實際申報日可能落在 5 月。
若以期末日對齊，模型在 4 月就用到了 5 月才公開的數字——
這是回測中最常見、也最難察覺的前視偏誤來源之一。

本研究一律以 XBRL 欄位中的**申報日**為準。

:::


## 滾動回測設計

| 參數 | 設定 |
| :--- | :---: |
| 形成期長度 | **252** 個交易日（約一年） |
| 交易期長度 | **126** 個交易日（約半年） |
| 滾動步長 | **21** 個交易日（約一月） |
| 同時重疊期數 | **6**（= 126 / 21） |

::: {.callout-important}

### 一項對統計設計有決定性影響的性質

滾動步長 **小於** 交易期長度 → 任一時點有 **6 個交易期同時運行**
→ **逐期報酬序列存在結構性自相關**。

此性質決定了 §3.5 為何不能以「期」為抽樣單位，
以及為何主檢定必須使用能保住自相關結構的 **block bootstrap**。

:::

**交易成本**　單邊 **0.29%**（Do & Faff, 2012 之 ~30bps 估計），
進出場各扣一次 → 一往返約名目額 **0.58%**。
該估計樣本期為 1962–2009，套用於 2000–2025 **偏保守**；
第四章另報 break-even 成本。

<svg viewBox="0 0 900 340" width="100%" style="max-width:960px;font-family:Microsoft JhengHei, Noto Sans TC, Segoe UI, sans-serif">
  <text x="70" y="26" font-size="16" font-weight="600" fill="#222">形成期 252 日 ／ 交易期 126 日 ／ 每 21 日滾動一次</text>
  <rect x="70" y="58" width="250" height="20" fill="#cfd8e6" stroke="#8fa3bf"/><rect x="320" y="58" width="125" height="20" fill="#e8964f" opacity="0.85"/><text x="62" y="73" font-size="13" fill="#666" text-anchor="end">第 1 期</text><rect x="91" y="92" width="250" height="20" fill="#cfd8e6" stroke="#8fa3bf"/><rect x="341" y="92" width="125" height="20" fill="#e8964f" opacity="0.85"/><text x="83" y="107" font-size="13" fill="#666" text-anchor="end">第 2 期</text><rect x="112" y="126" width="250" height="20" fill="#cfd8e6" stroke="#8fa3bf"/><rect x="362" y="126" width="125" height="20" fill="#e8964f" opacity="0.85"/><text x="104" y="141" font-size="13" fill="#666" text-anchor="end">第 3 期</text><rect x="133" y="160" width="250" height="20" fill="#cfd8e6" stroke="#8fa3bf"/><rect x="383" y="160" width="125" height="20" fill="#e8964f" opacity="0.85"/><text x="125" y="175" font-size="13" fill="#666" text-anchor="end">第 4 期</text><rect x="154" y="194" width="250" height="20" fill="#cfd8e6" stroke="#8fa3bf"/><rect x="404" y="194" width="125" height="20" fill="#e8964f" opacity="0.85"/><text x="146" y="209" font-size="13" fill="#666" text-anchor="end">第 5 期</text><rect x="175" y="228" width="250" height="20" fill="#cfd8e6" stroke="#8fa3bf"/><rect x="425" y="228" width="125" height="20" fill="#e8964f" opacity="0.85"/><text x="167" y="243" font-size="13" fill="#666" text-anchor="end">第 6 期</text><rect x="196" y="262" width="250" height="20" fill="#cfd8e6" stroke="#8fa3bf"/><rect x="446" y="262" width="125" height="20" fill="#e8964f" opacity="0.85"/><text x="188" y="277" font-size="13" fill="#666" text-anchor="end">第 7 期</text>
  <line x1="445" y1="46" x2="445" y2="300" stroke="#c0392b" stroke-width="2.5" stroke-dasharray="6 4"/>
  <text x="455" y="44" font-size="14" font-weight="600" fill="#c0392b">任一交易日</text>
  <text x="455" y="316" font-size="14" font-weight="600" fill="#c0392b">此垂直線切過 6 條交易期長條 → 6 組部位同時在倉</text>
  <rect x="70" y="308" width="18" height="13" fill="#cfd8e6" stroke="#8fa3bf"/>
  <text x="94" y="319" font-size="13" fill="#222">形成期（選配對、估參數）</text>
  <rect x="290" y="308" width="18" height="13" fill="#e8964f" opacity="0.85"/>
  <text x="314" y="319" font-size="13" fill="#222">交易期（實際持有部位、產生逐日報酬）</text>
</svg>

::: {.aside}
**圖 3-1**　滾動設計與部位重疊。此重疊即 §3.5 逐日報酬差自相關的來源。
:::


# 3.2 形成期的四層架構

## 設計原理：讓單變因成為結構保證

配對交易的形成期通常被實作為**單一整體流程** →
更換任一環節時難以歸因其效果。

本研究拆解為四個**可獨立替換**的層：

<svg viewBox="0 0 950 210" width="100%" style="max-width:1000px;font-family:Microsoft JhengHei, Noto Sans TC, Segoe UI, sans-serif">
  <defs><marker id="ar" markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto">
    <path d="M0,0 L7,3 L0,6 z" fill="#666"/></marker></defs>
  <text x="30" y="30" font-size="15" fill="#222">四層皆可獨立替換；檢定某一層時，其餘三層參數完全相同</text>
  <rect x="30" y="60" width="185" height="96" rx="8" fill="#f4f6fa" stroke="#8fa3bf" stroke-width="1.5"/><text x="122" y="88" font-size="17" font-weight="600" fill="#222" text-anchor="middle">特徵萃取</text><text x="122" y="110" font-size="12.5" fill="#666" text-anchor="middle">個股 → 特徵向量</text><text x="122" y="132" font-size="12.5" fill="#c0392b" text-anchor="middle">19 維（連續 7）</text><line x1="221" y1="108" x2="259" y2="108" stroke="#666" stroke-width="1.6" marker-end="url(#ar)"/><text x="241" y="98" font-size="11.5" fill="#666" text-anchor="middle">介面</text><rect x="267" y="60" width="185" height="96" rx="8" fill="#f4f6fa" stroke="#8fa3bf" stroke-width="1.5"/><text x="359" y="88" font-size="17" font-weight="600" fill="#222" text-anchor="middle">分組</text><text x="359" y="110" font-size="12.5" fill="#666" text-anchor="middle">特徵 → 搜尋空間</text><text x="359" y="132" font-size="12.5" fill="#c0392b" text-anchor="middle">HDBSCAN／AGG／KM／GICS</text><line x1="458" y1="108" x2="496" y2="108" stroke="#666" stroke-width="1.6" marker-end="url(#ar)"/><text x="478" y="98" font-size="11.5" fill="#666" text-anchor="middle">介面</text><rect x="504" y="60" width="185" height="96" rx="8" fill="#f4f6fa" stroke="#8fa3bf" stroke-width="1.5"/><text x="596" y="88" font-size="17" font-weight="600" fill="#222" text-anchor="middle">群內排序</text><text x="596" y="110" font-size="12.5" fill="#666" text-anchor="middle">組內配對 → 優先序</text><text x="596" y="132" font-size="12.5" fill="#c0392b" text-anchor="middle">SSD／DTW／SSD-DTW-PCA</text><line x1="695" y1="108" x2="733" y2="108" stroke="#666" stroke-width="1.6" marker-end="url(#ar)"/><text x="715" y="98" font-size="11.5" fill="#666" text-anchor="middle">介面</text><rect x="741" y="60" width="185" height="96" rx="8" fill="#f4f6fa" stroke="#8fa3bf" stroke-width="1.5"/><text x="833" y="88" font-size="17" font-weight="600" fill="#222" text-anchor="middle">統計篩選</text><text x="833" y="110" font-size="12.5" fill="#666" text-anchor="middle">淘汰不合格配對</text><text x="833" y="132" font-size="12.5" fill="#c0392b" text-anchor="middle">ADF／半衰期／Hurst</text>
  <rect x="30" y="168" width="890" height="30" rx="6" fill="#fdf3e7" stroke="#e8964f"/>
  <text x="475" y="188" font-size="13.5" fill="#222" text-anchor="middle">→ 命題 1 只變動「分組」層　·　命題 2 固定全部四層，只換交易端</text>
</svg>

::: {.aside}
**圖 3-2**　形成期四層架構與各層可替換的選項。
:::

各層以標準化介面銜接：
特徵層輸出 $(N 	imes d)$ 矩陣 → 分組層輸出 $\{股票 	o 組標籤\}$
→ 排序層在組內選前 $N$ 組 → 篩選層對價差施加統計檢定。

（上圖為**介面**的相依順序；排序與篩選在實作上的**施行**次序依後端而異，見 §3.2 排序層與篩選層。）

::: {.callout-tip}

此架構使「單變因對照」成為**結構上的保證**，而非人為約定——
檢定分組方法時，其餘三層的參數完全相同。

:::


## 特徵層：19 維（連續 7 維）

| 區塊 | 維度 | 內容 | 依據 |
| :--- | :---: | :--- | :--- |
| 報酬主成分載荷 | **5** | 形成期日報酬 PCA 前 5 主成分載荷，以特徵值平方根加權 | Avellaneda & Lee (2010) |
| 公司基本面 | **2** | 對數市值、盈餘殖利率（1/PE） | — |
| GICS 產業 one-hot | **12** | 11 大產業 + 1 未知 | — |

各區塊**獨立標準化**後依權重拼接（避免 one-hot 欄位數稀釋連續特徵的距離量測）。

缺失值以**產業中位數**插補後 winsorize（1%/99%）。

> ⚠️ 第四章將指出：此插補方式構成一條**未被察覺的產業資訊管道**，
> 並引入全域中位數插補作為對照。


## 分組層：三種分群 + 對照組

| 方法 | 關鍵參數 | 群數決定方式 |
| :--- | :--- | :--- |
| **HDBSCAN** | `min_cluster_size`=5, `min_samples`=2 | 資料驅動，可標記噪音 |
| **Agglomerative** | average linkage，門檻取距離分布 **75 分位** | 由門檻決定 |
| **K-means** | $k$ = **同期 Agglomerative 的群數** | 對齊使量級可比 |
| **GICS（對照）** | — | 11 大產業，不跑分群 |

::: {.callout-note}

### K-means 的群數為何要對齊

K-means 需**預先指定**群數，若任意給定將使其與其他方法不可比。
本研究先跑一次 Agglomerative 取得資料驅動的群數再餵給 K-means，
確保比較聚焦於**分群機制**而非**粒度**。

:::

HDBSCAN 的噪音點（標籤 −1）與過小群（成員 < 5）併入「Unknown」，於排序層跳過。


## 排序層與篩選層

### 排序層：三種距離準則

| 準則 | 定義 |
| :--- | :--- |
| **SSD** | 正規化對數價格路徑的平方差總和（GGR, 2006） |
| **DTW** | 動態時間校正，Sakoe-Chiba 頻帶寬 15（許鈞翔, 2025） |
| **SSD-DTW-PCA** | 兩距離的主成分融合，取第一主成分 |

### 篩選層：三道檢定（任一未過即淘汰）

1. **ADF 共整合**　殘差 $p < 0.05$（Engle & Granger, 1987）
2. **OU 半衰期**　$HL = -\ln 2 / \lambda$，要求 $1 \le HL \le 42$ 日
3. **Hurst 指數**　$H < 0.5$（均值回歸傾向）

半衰期上限 42 日 = 交易期 126 日的 1/3，確保價差有足夠時間回歸。

::: {.callout-important}

### 排序與篩選的施行次序依後端而異

| 後端 | 實際流程 |
| :--- | :--- |
| **SSD** | 先按 SSD 排序，取前 $C = \maxigl(200,\; 15\,Nigr)$ 組為候選池 → **依距離由近而遠**逐一施加篩選 → 累積通過數達 $5N$ 即提前中止 → 自通過者中取距離最小的前 $N$ 組 |
| **DTW／SSD-DTW-PCA** | 群內**全配對**逐一施加篩選（無候選上限、不提前中止）→ 自通過者中排序取前 $N$ 組 |

其中 $N$ = `top_n`（每期目標配對數，網格值 5／10／20）。

兩者的**共同**後果相同，且是理解第四章結果的關鍵：
**候選池不足時，系統會被迫接受距離更遠的配對**——SSD 因候選上限與提前中止而更早觸發，
DTW 則因通過篩選者不足 $N$ 組而觸發。

:::

::: {.callout-note}

### 這是「可獨立替換」的一項例外，須明確揭露

四層架構使分組層的替換成為乾淨的單變因對照。但**排序層的替換同時改變了
排序與篩選的施行次序**（上表），故排序層之間的比較嚴格說並非單一變因。

本研究的主命題落在**分組層**（命題 1）與**交易端**（命題 2），
兩者皆固定排序後端，不受此影響；涉及排序層的比較僅作描述性報告。

:::


# 3.3 分群的逐期重估與其實測

本研究的分群模型於**每一形成期獨立重新配適**，而非全樣本分群一次後固定。
本節界定此一設計，並以兩項指標實測其「逐期重估」是否確實產生了不同的分群結果。

| 性質 | 內容 |
| :--- | :--- |
| **逐期重估** | 每期以該期 252 日視窗重建特徵、重新配適；不保留跨期狀態 |
| **重估次數** | **295 期**（2000-01-03 → 2024-07-01），三種分群法各配適 295 次 |
| **群數隨資料變動** | HDBSCAN 由密度決定；Agglomerative 取當期距離 75 分位；K-means 對齊之 |
| **前視偏誤防範** | 全樣本分群會讓 2005 年的搜尋空間受 2020 年共變影響 |


## 動態性的實測①：配對層級不具鑑別力

| 分組方法 | 相鄰期配對重疊率 | 配對存續期數（中位） |
| :--- | ---: | ---: |
| Agglomerative | 9.8% | 1 |
| HDBSCAN | 9.4% | 1 |
| K-means | 6.4% | 1 |
| **GICS（靜態對照）** | **11.7%** | 1 |

::: {.callout-warning}

### 靜態的 GICS 週轉率反而最高

配對週轉主要由**排序層與篩選層**驅動——即使分組固定，
每期的距離排序與共整合檢定結果本就不同。
**此指標無法佐證分群層的動態性。**

:::


## 動態性的實測②：分群層級以 ARI 量測

重跑形成期前兩層取出群標籤，計算相鄰期在**共同標的**上的
**調整蘭德指數**（群編號無意義，故比對「兩兩是否同群」；ARI 已對隨機一致校正）。

| 分組方法 | 群數 | 相隔 1 期（21 日） | 相隔 6 期（126 日） |
| :--- | ---: | ---: | ---: |
| Agglomerative | 104 | **0.709** | 0.483 |
| HDBSCAN | 16 | **0.685** | 0.498 |
| K-means | 104 | **0.515** | 0.350 |
| **GICS（靜態）** | 11 | **1.000** | 1.000 |

- **結構確實逐期改變**：相隔 21 日已有三至五成分群關係改變
- **改變隨時間累積**：一個交易期走完，過半結構已不同
- **K-means 最不穩定**（隨機初始化；Agglomerative 的階層合併較不敏感）

> **範圍聲明**：確立分群**確實在變**，但**未**檢定這種改變是否**有益**——
> 本研究無「靜態分群」對照組。逐期重估是前視偏誤防範的必要設計，
> 而非受檢定的處理。


## 消融矩陣：4 分組 × 3 排序

## 4 分組 × 3 排序消融矩陣

|  | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| **GICS（對照）** | ✓ | ✓ | ✓ |
| HDBSCAN | ✓ | ✓ | ✓ |
| Agglomerative | ✓ | ✓ | ✓ |
| K-means | ✓ | ✓ | ✓ |

固定特徵層、篩選層與交易端 → **唯一變因為分組方法**。
每格再展開 `top_n` × `stop_loss`（5 × 3 = 15 種配置）。

同一排序準則下的 ML 分群與 GICS 構成直接對照，共 **9 組**（3 分群 × 3 排序）。

### 三項受控消融（用於失敗歸因）

| 消融 | 參數 | 檢驗什麼 |
| :--- | :--- | :--- |
| **產業先驗強度** | `sector_onehot_weight` ∈ {1.0, 0} | one-hot 對跨產業配對的距離懲罰 |
| **統計篩選** | `filter_mode` ∈ {coint, none} | 篩選的貢獻，及其與分群的**交互作用** |
| **分組維度零點** | `cluster_method` = **none** | 「限制搜尋空間」本身的價值 |

> 若無「不分組」對照，所比較的僅是不同的**限制方式**，而非限制本身。


# 3.4 門檻選擇的深度學習實作（命題 2）

## 動作空間：門檻選擇式而非逐日定位

**基準（對照臂）**　Z-Score 規則：$\lvert z\rvert > 2.0$ 進場、$z$ 穿越 0 平倉、
期末強平、另設停損。整段交易期沿用同一組門檻，不隨市況調整。

**待檢定臂（DL-THR）**　每組配對、每個交易期**只做一次決策**：
自 9 個動作中選一個，選定後交由**完全相同**的 Z-Score 狀態機執行整期。

### 9 個動作是怎麼來的

$$\mathcal{A} = \{\text{SKIP}\} \;\cup\; \bigl\{(\text{entry\_z},\ \text{exit\_z})\bigr\}
= 1 + 4 \times 2 = 9$$

| 參數 | 候選值 | 含義 |
| :--- | :--- | :--- |
| **entry\_z** | 1.5, 2.0, 2.5, 3.0 | 價差偏離多少個標準差才進場。**越大 = 越保守**，等待更極端的偏離 |
| **exit\_z** | 0.0, 0.5 | 價差回歸到多少才平倉。0.0 = 等回到均值；**0.5 = 提前獲利了結**，不等完全收斂 |
| **SKIP** | — | 本期**完全不交易**該組配對，資金釋放給其他配對 |

四個 entry 值涵蓋了自寬鬆（1.5）到嚴格（3.0）的實務常見範圍，
基準值 2.0 落在其中；兩個 exit 值則對比「等完全回歸」與「見好就收」兩種出場哲學。

::: {.callout-important}

### 三項結構性保證

1. **選單含靜態基準 $(2.0,\ 0.0)$** → 待檢定臂的策略空間**必然包含**對照臂。
   模型最差的情況是永遠選這一格，此時兩臂完全相同、效果量為零——
   **設計上不可能因「模型亂選」而系統性劣於基準**。
2. **訓練樣本不足時強制選基準動作** → 樣本初期兩臂行為**逐位元相同**，
   避免暖身期的隨機表現污染整段比較。
3. **SKIP 提供拒絕權** → 固定規則不具備的選擇性。
   此項是增益的候選來源之一，故第四章對它另設置換檢定（§4.2.3）。

:::

::: {.callout-note}

### 為何不用逐日定位的動作空間

一個更「自然」的設計是讓模型每日自由決定持倉方向與大小。
本研究曾實作三代此類版本，結果模型對**日級噪音計時**，
換手成本大幅上升、績效顯著劣於基準（中位 Sharpe −1.1 至 −2.3）。

失敗根因可被隔離於**動作空間設計**而非學習演算法——
因為改成「每期選一次門檻」後，同一套網路與特徵即產生正向且顯著的增益。
此對比是第五章「關鍵是動作空間設計，不是訓練方法」該項結論的來源之一。

:::


## 狀態空間與學習問題的性質

**狀態 = 12 維形成期特徵**（全部可於交易期開始前計算）

| 類別 | 特徵 |
| :--- | :--- |
| 偏離狀態 | 期末 $z$、期末 $\|z\|$ |
| 回歸品質 | 零穿越頻率、OU 半衰期對數 |
| 近期 regime | 近 21 日 $z$ 波動相對全期、近 21 日 $z$ 趨勢 |
| 配對性質 | 兩檔報酬相關係數、波動比、對沖比例偏離 1 的幅度 |
| 可交易性 | 價差振幅、形成期 $\|z\|>2$ 佔比、形成期最大 $\|z\|$ |

::: {.callout-tip}

### 這是全資訊監督回歸，不是 bandit

歷史配對期中，**全部 9 個動作的報酬都可精確反事實回算**
（對該期價格逐一模擬 9 組門檻）→ **無探索問題、樣本效率最高**。

網路：MLP（12 → 隱藏 64 → 9 個動作報酬），MSE 損失，每期增量訓練 40 epoch。

:::

::: {.callout-warning}

### 命名說明

本研究早期版本及程式碼中稱此交易端為「**DRL**」（deep reinforcement learning）。
依上述，其學習問題為**全資訊監督回歸**而非強化學習——動作報酬可完整反事實回算，
不存在探索與利用的取捨。為避免與真正的部分回饋設定混淆，
本文一律稱其為 **DL-THR**（deep-learning threshold selection）。

此區分具實質意義：下一張的受控對照刻意實作了一個**真正的** contextual bandit
版本（**RL-THR**），若主臂仍沿用「DRL」之名，該節將無法閱讀。

> `result.db` 的 strategy id（如 `Grid (AGG-SSD-DRL)`）為**歷史命名**，
> 未隨本文更動，以維持與既有回測輸出的可對照性。

:::


## 前視偏誤的防範與隨機性處理

::: {.callout-important}

### walk-forward 增量訓練

> 期 $k$ 的決策，**僅使用「交易期已於期 $k$ 開始前結束」的樣本**訓練。

實作為對訓練緩衝區施加 `trade_end < trade_start_k` 過濾。
可用樣本 < 200 筆時，自動選用基準動作 $(2.0, 0.0)$。

:::

**隨機性處理**　網路未固定隨機種子 → 對三種 ML 配對底各執行**五輪獨立重訓**，
以中位數與全距報告，避免單次訓練的隨機性被誤讀為方法效果。


## 受控對照：把全資訊標籤換成部分回饋

如前所述，**DL-THR 不是強化學習**：9 個動作的報酬在每個歷史配對期上都能精確
反事實回算，模型拿到的是完整答案卷，學習形式為全資訊監督回歸
（MLP + MSE，決策取 $\arg\max$），既無探索問題、亦無序列信用分配。

為分離「學習演算法」與「這個問題碰巧是全資訊的」兩件事，本研究另建 **RL-THR**
作為受控對照。它保持動作選單、12 維狀態、網路結構與 walk-forward 切分**逐位元相同**
（特徵函式直接引用同一份實作），只改兩處：

| 環節 | DL-THR | RL-THR |
| :--- | :--- | :--- |
| 訓練標籤 | 9 個動作全部反事實回算 | **只有實際選中的那一個** |
| 損失 | 全 9 維 MSE | **遮罩 MSE**（僅選中動作的輸出單元收梯度） |
| 決策 | 恆 $\arg\max$ | **$\varepsilon$-greedy** |

於是 RL-THR 成為真正的部分回饋問題，兩者相減即為**反事實標籤的價值**。

::: {.callout-important}

### 形式歸屬：contextual bandit，沒有 $\gamma$

RL-THR 是**情境式拉霸機**而非序貫 MDP。每組配對每期只做一次決策，
且 12 維狀態全部由形成期視窗算出——選哪個門檻**不會改變下一期的狀態**。
沒有狀態轉移就沒有東西可以 bootstrap，硬加折扣因子只是裝飾。
其「RL 性」來自部分回饋與探索／利用權衡（Sutton & Barto, 2018, 第 2 章），
而非 TD 學習。本研究不宣稱序貫決策。

真正的序貫版本（逐日定位、$\gamma$=0.99、bootstrapped target）已於三代實作中
系統性證偽，見 §3.4 註與封存紀錄。

:::

::: {.callout-warning}

### RL-THR 少一項結構性保證，且兩個效應無法分離

DL-THR 保證「樣本不足時退回基準 → 暖身期 $\equiv$ Z-Score」。RL-THR **無法**如此：
若暖身期一律選基準，其餘八個動作永遠沒有樣本。探索必須從第一期開始。

因此兩臂的差距同時包含**資訊量**（每期 1 筆觀測 vs 9 筆）與**探索成本**
（$\varepsilon$ 抽中時實際下單、虧損計入績效）。本設計無法拆解為兩個獨立數字——
不探索就沒有樣本，這是部分回饋的定義。掃三組 $\varepsilon$
（$0.05$、$0.10$、$0.20\!\to\!0.02$ 衰減）以界定取捨曲線，
並以**對 bandit 最有利者**作結論，使推論保守。

:::


# 3.5 統計檢定方法

## 抽樣單位：為何不能用參數網格

本研究早期版本以「參數網格」為抽樣單位——對 15 種 `top_n` × `stop_loss`
配置作配對 $t$ 檢定。

::: {.callout-important}

### 偽重複（pseudo-replication）

15 個「觀測」共用**同一份資料、同一段期間、同一批配對**：

- `top_n`=10 與 `top_n`=20 **共用 10 組配對**
- 三種停損是**同一批交易**的不同出場規則

觀測間高度相關 → 不滿足 $t$ 檢定的獨立性假設 →
**有效樣本數接近一條回測路徑，而非 15**。

:::

**本研究改以「時間」為抽樣單位。**


## 主檢定（一）：兩個零件各自在做什麼

主檢定的全名是「**逐日報酬差 + 循環 block bootstrap**」。這是兩個獨立的零件，
分別解決兩個不同的問題。

### 零件一：逐日報酬差 —— 解決「拿什麼跟什麼比」

$$\Delta r_t \;=\; r_{	ext{處理},t} \;-\; r_{	ext{對照},t}, \qquad t = 1, \dots, 6287$$

每個交易日，把待檢定策略與對照策略的當日報酬**相減**，得到一條長度 6,287 的差分序列。

::: {.callout-important}

### 為何要相減，而不是各自算績效再比

兩策略在同一天承受**同一批市場衝擊**——2008 年金融海嘯、2020 年三月熔斷，
兩邊都會虧。這些共同成分是**噪音**，它對「哪個方法比較好」這個問題不提供資訊，
卻主宰了報酬的變異。

相減即可**消去共同成分**，剩下的才是兩個方法之間的差異。
這是配對設計（paired design）的標準作法，也是命題 2 的檢定力遠高於絕對績效
檢定的原因（§4.6.4）。

**未持倉日記為 0，而非遺漏值**——該日策略確實沒有部位，這是它的行為的一部分；
視為遺漏會系統性排除「不交易」這個決策。

:::

### 零件二：循環 block bootstrap —— 解決「差分序列不獨立」

$\Delta r_t$ 逐日之間**並不獨立**：滾動步長 21 日小於交易期 126 日，
任一時點有 6 個交易期同時運行，同一批部位的損益會連續出現好幾天（§3.1 滾動設計）。

一般的統計檢定假設觀測獨立。若直接套用，等於把 6,287 個高度相關的觀測
當成 6,287 個獨立樣本，**標準誤會被低估、$p$ 值會被高估其顯著性**。

**block bootstrap 的作法**：不抽單一天，改抽**連續的一段**。

<svg viewBox="0 0 900 300" width="100%" style="max-width:960px;font-family:Microsoft JhengHei, Noto Sans TC, Segoe UI, sans-serif">
  <text x="70" y="30" font-size="15" fill="#222">① 原始逐日差分序列（6,287 日）——首尾相接成「環」，使每一日被抽中的機率相等</text>
  <rect x="70" y="56" width="58" height="30" fill="#e8eef7" stroke="#8fa3bf"/><rect x="128" y="56" width="58" height="30" fill="#dbe4f0" stroke="#8fa3bf"/><rect x="186" y="56" width="58" height="30" fill="#e8eef7" stroke="#8fa3bf"/><rect x="244" y="56" width="58" height="30" fill="#dbe4f0" stroke="#8fa3bf"/><rect x="302" y="56" width="58" height="30" fill="#e8eef7" stroke="#8fa3bf"/><rect x="360" y="56" width="58" height="30" fill="#dbe4f0" stroke="#8fa3bf"/><rect x="418" y="56" width="58" height="30" fill="#e8eef7" stroke="#8fa3bf"/><rect x="476" y="56" width="58" height="30" fill="#dbe4f0" stroke="#8fa3bf"/><rect x="534" y="56" width="58" height="30" fill="#e8eef7" stroke="#8fa3bf"/><rect x="592" y="56" width="58" height="30" fill="#dbe4f0" stroke="#8fa3bf"/><rect x="650" y="56" width="58" height="30" fill="#e8eef7" stroke="#8fa3bf"/><rect x="708" y="56" width="58" height="30" fill="#dbe4f0" stroke="#8fa3bf"/>
  <path d="M70,71 C40,71 40,101 70,101" fill="none" stroke="#666" stroke-width="1.5" stroke-dasharray="4 3"/>
  <path d="M766,71 C796,71 796,101 766,101" fill="none" stroke="#666" stroke-width="1.5" stroke-dasharray="4 3"/>
  <text x="418" y="112" font-size="12.5" fill="#666" text-anchor="middle">環狀：序列末端接回開頭</text>
  <text x="70" y="150" font-size="15" fill="#222">② 隨機挑起點，整段取出長度 L 的連續區塊（區塊內部的自相關被完整保留）</text>
  <rect x="186" y="56" width="116" height="30" fill="#f6c89a" stroke="#c0392b" stroke-width="2"/><rect x="70" y="188" width="180" height="30" fill="#f6c89a" stroke="#c0392b" stroke-width="2"/><text x="160" y="208" font-size="12.5" fill="#222" text-anchor="middle">區塊 1（L=126 日）</text><rect x="476" y="56" width="116" height="30" fill="#f0a868" stroke="#c0392b" stroke-width="2"/><rect x="270" y="188" width="180" height="30" fill="#f0a868" stroke="#c0392b" stroke-width="2"/><text x="360" y="208" font-size="12.5" fill="#222" text-anchor="middle">區塊 2（L=126 日）</text><rect x="302" y="56" width="116" height="30" fill="#e8964f" stroke="#c0392b" stroke-width="2"/><rect x="470" y="188" width="180" height="30" fill="#e8964f" stroke="#c0392b" stroke-width="2"/><text x="560" y="208" font-size="12.5" fill="#222" text-anchor="middle">區塊 3（L=126 日）</text>
  <text x="70" y="248" font-size="15" fill="#222">③ 接成等長的新序列 → 重複 10,000 次 → 得到平均差分的經驗分布</text>
  <text x="70" y="276" font-size="13" fill="#c0392b">只有區塊「之間」被打散；6 期重疊造成的日間相關性留在區塊內，故標準誤不被低估</text>
</svg>

::: {.aside}
**圖 3-3**　循環 block bootstrap 的重抽程序。
:::


區塊**內部**的自相關結構被原封不動保留，只有區塊**之間**被打散。
「循環」是指把序列首尾相接，使每個觀測被抽中的機率相等
（否則頭尾的觀測會被系統性低估）。


## 主檢定（二）：判準與依據

### 由重抽分布得到兩個東西

以 10,000 次重抽得到 10,000 個「假想的平均差分」，構成經驗分布：

| 輸出 | 作法 | 判準 |
| :--- | :--- | :--- |
| **$p$ 值** | 將分布**平移使中心為零**（即強加 $H_0$：兩策略無差異），再看實際觀測到的平均差分落在何處 | 雙尾 $p < 0.05$ 判定為顯著 |
| **95% 信賴區間** | 取**未平移**分布的 2.5 與 97.5 百分位 | 區間是否涵蓋 0；**以及區間有多寬** |

$H_0$：$E[\Delta r] = 0$（兩交易端／兩分組法的期望日報酬相同）。

::: {.callout-important}

### 判準不只是「$p < 0.05$」——區間寬度承擔同等份量

本研究對「不顯著」一律追問一步：

| 情形 | 讀法 |
| :--- | :--- |
| 區間**不含 0** | 方向性主張成立 |
| 區間含 0，且**兩端都小**（相對於參照臂自身的績效量級） | 效果縱使存在也不具實質意義 → 可討論「**相當**」 |
| 區間含 0，但**兩端都大** | 資料無法區分 → **檢定力不足**，不可宣稱相當 |

命題 1 屬第三種、§4.2.3 的 RL-THR 對照屬第二種——
**同樣是「不顯著」，結論相反**。若只報 $p$ 值，這個區別會完全消失。

:::

### 三項參數為何如此設定，而非搜尋而得

| 參數 | 值 | 依據 |
| :--- | :--- | :--- |
| 區塊長度 $L$ | **126** | ＝一個完整交易期（`FORWARD_DAYS`）。由設計直接給定，**非**調參結果，故不作 $L$ 敏感度分析 |
| 重抽次數 | **10,000** | 使 $p$ 值的蒙地卡羅誤差在 0.05 附近約 ±0.004，遠小於判準本身 |
| 顯著水準 | **5%** | 慣例；多組對照時另以 Benjamini-Hochberg 控制 FDR |

::: {.callout-note}

### 為何不採 Newey-West HAC

兩法全程並行計算，**14 組對照結論完全一致**。換用的理由不在結果而在**可辯護性**：

| | HAC | block bootstrap |
| :--- | :--- | :--- |
| 分布假設 | 漸近常態 | 無 |
| 研究者自由度 | **落後階**須自選並辯護 | 無（$L$ 由交易期長度給定） |
| 日報酬厚尾偏態 | 近似可能失準 | 不受影響 |
| 輸出 | $p$ 值 | $p$ 值**與**信賴區間 |

HAC 保留為各表的**對照欄**。方法依據見 §2.4。

:::

<svg viewBox="0 0 900 300" width="100%" style="max-width:960px;font-family:Microsoft JhengHei, Noto Sans TC, Segoe UI, sans-serif">
  <text x="20" y="30" font-size="15" fill="#222">同樣是「區間涵蓋 0」，結論可以相反——差別在<tspan font-weight="600">兩端離 0 多遠</tspan></text>
  <line x1="300" y1="52" x2="300" y2="272" stroke="#222" stroke-width="1.6"/>
  <text x="300" y="48" font-size="13" fill="#222" text-anchor="middle">0</text>
  <text x="20" y="97" font-size="13.5" font-weight="600" fill="#222">命題 2（交易期）</text><line x1="326.0" y1="92" x2="528.8" y2="92" stroke="#2e7d32" stroke-width="7" stroke-linecap="round"/><line x1="326.0" y1="83" x2="326.0" y2="101" stroke="#2e7d32" stroke-width="2.5"/><line x1="528.8" y1="83" x2="528.8" y2="101" stroke="#2e7d32" stroke-width="2.5"/><text x="318.0" y="97" font-size="12" fill="#666" text-anchor="end">+0.20</text><text x="536.8" y="97" font-size="12" fill="#666">+1.76</text><text x="20" y="118" font-size="12.5" fill="#2e7d32">區間完全在 0 右側 → 方向性主張成立</text><text x="20" y="165" font-size="13.5" font-weight="600" fill="#222">RL-THR vs DL-THR</text><line x1="250.6" y1="160" x2="358.5" y2="160" stroke="#1565c0" stroke-width="7" stroke-linecap="round"/><line x1="250.6" y1="151" x2="250.6" y2="169" stroke="#1565c0" stroke-width="2.5"/><line x1="358.5" y1="151" x2="358.5" y2="169" stroke="#1565c0" stroke-width="2.5"/><text x="242.6" y="165" font-size="12" fill="#666" text-anchor="end">-0.38</text><text x="366.5" y="165" font-size="12" fill="#666">+0.45</text><text x="20" y="186" font-size="12.5" fill="#1565c0">含 0 且兩端皆小 → 可主張「相當」</text><text x="20" y="233" font-size="13.5" font-weight="600" fill="#222">命題 1（形成期）</text><line x1="173.9" y1="228" x2="422.2" y2="228" stroke="#c0392b" stroke-width="7" stroke-linecap="round"/><line x1="173.9" y1="219" x2="173.9" y2="237" stroke="#c0392b" stroke-width="2.5"/><line x1="422.2" y1="219" x2="422.2" y2="237" stroke="#c0392b" stroke-width="2.5"/><text x="165.9" y="233" font-size="12" fill="#666" text-anchor="end">-0.97</text><text x="430.2" y="233" font-size="12" fill="#666">+0.94</text><text x="20" y="254" font-size="12.5" fill="#c0392b">含 0 且兩端皆大 → 檢定力不足，不可稱相當</text>
  <text x="300" y="292" font-size="12.5" fill="#666" text-anchor="middle">年化報酬差（百分點，pp）</text>
</svg>

::: {.aside}
**圖 3-4**　信賴區間的三種判讀；數值取自第四章實際結果。
:::


## 信賴區間、多重檢定與報告口徑

### 信賴區間取代非劣性檢定

「無顯著差異」**不等於**「兩者相當」。區分兩者只需看**區間寬度**：

- 區間涵蓋 0 且**兩端皆小** → 效果縱使存在也不具實質意義
- 區間涵蓋 0 但**兩端皆大** → **檢定力不足**，資料無法區分

命題 1 即屬後者。舊版以非劣性檢定處理同一問題，須事前指定並辯護
容忍邊界 $\delta$；CI 承載相同資訊且不需此判斷，故改採 CI。

### 多重檢定校正

同一命題涉及多組對照時（如命題 1 的 9 組），
以 **Benjamini-Hochberg** 控制 FDR，同時報告原始與校正後 $p$ 值。

::: {.callout-important}

### 報告口徑：一律全網格等權組合

一切績效主張與統計檢定，皆以該策略 **15 個參數配置的等權組合**為口徑。

改報「網格最佳格」會內含 15 選 1 的選擇偏誤，必須再以 Deflated Sharpe 之類的
程序扣回去——而該程序的關鍵參數（試驗數 $N$）是判斷而非事實。
**等權組合沒有東西可挑，選擇偏誤自源頭消失。**

代價是絕對績效數字低於最佳格。這正是誠實的代價：
研究者事前並不知道哪一格會勝出。

此原則**無例外**，包含 §4.6 的 regime 分層與 break-even 成本表。

:::


## 報告口徑（續）：為何最佳格不能用於兩臂比較

網格最佳格常被用來刻畫單一策略的「風險形狀」，看似無害。
但只要比較的是**兩個臂**，該口徑就會引入兩個獨立的問題。

::: {.callout-important}

### 問題一：兩臂的最佳格通常不是同一格

各臂自行取最佳時，選到的參數配置往往不同。
本研究實測：`Grid (HDB-SDP)` 的最佳格為 `Top3/停損0%`，
而其 DL-THR 對應臂為 `Top1/停損0%`。

於是「哪一臂較好」的比較，同時混入了**交易端**與**參數格**兩個變因
——這正是 §3.2 的四層架構要排除的東西。

### 問題二：兩臂各自吃了一次選擇偏誤

自 15 格中挑最大值，本身即是一次 15 選 1。兩臂各挑一次，
差分中因而含有**兩份**方向不定的選擇偏誤，且無法互相抵銷。

:::

**本研究的作法**：§4.6 兩張表同樣以 15 格等權計算，且**逐格對齊兩臂**
——只取兩個交易端都存在的參數格，各自等權平均後才相比。
如此差分的唯一變因仍是交易端，與 §4.2 主檢定的口徑一致。

::: {.callout-note}

### 代價要說清楚

等權口徑下的 break-even 成本會**明顯低於**最佳格口徑。這不是計算變嚴，
而是最佳格原本就把「挑到最賺那一格」的運氣算進了成本承受度。

等權組合的絕對報酬本就與零無法區分（§4.6.3），
其 break-even 理當**貼著成本假設本身**——若它高出一到兩成，
反而應該懷疑該數字內含了選擇偏誤。

:::


## 本章小結

::: {.callout-important}

本研究的方法設計圍繞一項原則：

> **任兩個待比較的策略之間，僅存在單一變因。**

- **形成期**：四層架構使此原則成為**結構上的保證**
- **交易期**：利用「可在同一批配對上施行」的性質達成同樣效果

統計方面，以**時間**為抽樣單位取代參數網格，
以 **block bootstrap** 為單一主檢定（同時給出 $p$ 值與信賴區間），
並一律以**全網格等權組合**為報告口徑——
使推論不依賴分布假設，也不留下「挑最佳配置」的空間。

:::
